# Whole-brain searchlight MVPA

For each subject, computes a whole-brain decoding accuracy map for selected pairwise condition comparisons. Each voxel's value = leave-one-run-out CV accuracy of a linear SVM trained on patterns within an 8 mm sphere around that voxel.

Output: one NIfTI per subject per pair, e.g. `sub-102_searchlight_lowSF-lowC_vs_highSF-highC.nii.gz`.

After running this, group-compare the accuracy maps (BDD vs HC) — see the second-level cell at the bottom.

**Run time**: ~5–20 min per subject per pair, depending on CPU. With 59 subjects × 3 pairs, expect this to run overnight on a laptop. Reduce `PAIRS` to just the hypothesis-relevant comparison if you want a faster first pass.

## 1. Install / import

In [ ]:
%pip install nilearn nibabel scikit-learn pandas joblib scipy --quiet

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
import pandas as pd
from scipy.special import gamma as gamma_fn
from nilearn.decoding import SearchLight
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from datetime import datetime

## 2. Configuration

In [ ]:
# --- Paths ---------------------------------------------------------------
FMRIPREP_DIR = Path("/Volumes/drive/AVP-BDD/derivatives")
BEHAV_DIR    = Path("/Volumes/drive/AVP-BDD/behavior")
OUT_DIR      = Path("/Volumes/drive/AVP-BDD/derivatives/mvpa_searchlight")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Acquisition / task --------------------------------------------------
TR_SECONDS = 1.0
TASK       = "SFlow"
SPACE      = "MNI152NLin2009cAsym"

CONDITION_MAP = {
    "Condition 1": "lowSF-highC",
    "Condition 2": "highSF-highC",
    "Condition 3": "lowSF-lowC",
    "Condition 4": "highSF-lowC",
}

# --- Searchlight parameters ----------------------------------------------
RADIUS_MM = 8.0      # ~8 mm radius -> ~33 voxels at 3 mm
N_JOBS    = -1       # -1 = all CPU cores

# --- Pairwise comparisons -------------------------------------------------
PAIRS = [
    ("lowSF-lowC", "highSF-highC"),    # cross-diagonal: most relevant to hypothesis
    ("lowSF-lowC", "highSF-lowC"),     # SF effect within lowC
    ("lowSF-lowC", "lowSF-highC"),     # contrast effect within lowSF
]

# --- Subjects -------------------------------------------------------------
GROUP_1_SUBS = [s for s in range(102, 133) if s != 114]    # BDD, n=30
GROUP_2_SUBS = [s for s in range(201, 231) if s != 203]    # HC,  n=29
RUNS = [1, 2, 3]

print(f"BDD: {len(GROUP_1_SUBS)} subjects")
print(f"HC:  {len(GROUP_2_SUBS)} subjects")
print(f"Pairs to decode: {len(PAIRS)}")
print(f"Output: {OUT_DIR}")

## 3. HRF + design matrix helpers

In [ ]:
def double_gamma_hrf(t, a1=6.0, b1=1.0, a2=16.0, b2=1.0, c=1/6.0):
    """SPM-style canonical double-gamma HRF."""
    h = ((t**a1 * np.exp(-t/b1)) / (b1**(a1+1) * gamma_fn(a1+1))
         - c * (t**a2 * np.exp(-t/b2)) / (b2**(a2+1) * gamma_fn(a2+1)))
    h[t < 0] = 0
    return h


def build_lsa_design(behav_csv, n_tp, tr):
    """Least-squares-all design: one HRF-convolved column per BLOCK."""
    df = pd.read_csv(behav_csv)
    blocks = (df.groupby("Block")
                .agg(onset=("Stimulus Onset (s)",  "min"),
                     offset=("Stimulus Offset (s)", "max"),
                     condition=("Condition",        "first"))
                .reset_index().sort_values("onset"))

    fine_dt  = 0.05
    duration = n_tp * tr
    hrf      = double_gamma_hrf(np.arange(0, 32, fine_dt))
    n_fine   = int(duration / fine_dt)

    X = np.zeros((n_tp, len(blocks)))
    labels = []
    for col, (_, b) in enumerate(blocks.iterrows()):
        stick = np.zeros(n_fine)
        stick[int(b.onset/fine_dt):int(b.offset/fine_dt)] = 1.0
        conv = np.convolve(stick, hrf)[:n_fine]
        tr_idx = ((np.arange(n_tp) + 0.5) * tr / fine_dt).astype(int)
        tr_idx = np.clip(tr_idx, 0, n_fine - 1)
        X[:, col] = conv[tr_idx]
        labels.append(CONDITION_MAP[b.condition])
    return X, labels


def find_bold(sub_id, run):
    sub_root = FMRIPREP_DIR / sub_id
    glob = f"**/{sub_id}*_task-{TASK}_run-{run:02d}_space-{SPACE}_desc-preproc_bold.nii.gz"
    hits = list(sub_root.glob(glob))
    return hits[0] if hits else None


def find_brain_mask(sub_id, run):
    sub_root = FMRIPREP_DIR / sub_id
    glob = f"**/{sub_id}*_task-{TASK}_run-{run:02d}_space-{SPACE}_desc-brain_mask.nii.gz"
    hits = list(sub_root.glob(glob))
    return hits[0] if hits else None

## 4. Per-subject block-level beta extraction

In [ ]:
def build_subject_betas(sub_num):
    """Fit voxelwise OLS with the LSA design; return 4D beta image + labels + run IDs."""
    sub_id = f"sub-{sub_num}"
    raw_id = str(sub_num)

    all_betas, all_labels, all_runs, ref_img = [], [], [], None
    for run in RUNS:
        bold  = find_bold(sub_id, run)
        behav = BEHAV_DIR / raw_id / "low-level" / f"Subject_{raw_id}_Run_{run}_RT.csv"
        if bold is None or not behav.exists():
            print(f"  {sub_id} run-{run}: missing files, skipping"); continue

        img  = nib.load(str(bold))
        if ref_img is None:
            ref_img = img
        data = img.get_fdata(dtype=np.float32)
        n_tp = data.shape[3]

        # Z-transform per voxel time course
        m = data.mean(axis=3, keepdims=True); s = data.std(axis=3, keepdims=True)
        s[s == 0] = 1
        data = (data - m) / s

        X, labels = build_lsa_design(behav, n_tp, TR_SECONDS)

        # Voxelwise OLS, vectorized:  β = (XᵀX)⁻¹ Xᵀ y
        XtX_inv_Xt = np.linalg.pinv(X.T @ X) @ X.T            # (n_blocks, T)
        flat_ts    = data.reshape(-1, n_tp).T                  # (T, n_voxels)
        flat_betas = XtX_inv_Xt @ flat_ts                      # (n_blocks, n_voxels)
        betas      = flat_betas.T.reshape(*data.shape[:3], -1) # (X, Y, Z, n_blocks)

        all_betas.append(betas)
        all_labels.extend(labels)
        all_runs.extend([run] * len(labels))

    if not all_betas:
        return None, None, None

    full = np.concatenate(all_betas, axis=3)
    return (nib.Nifti1Image(full, ref_img.affine, ref_img.header),
            np.array(all_labels), np.array(all_runs))

## 5. Searchlight runner

In [ ]:
def run_searchlight_subject(sub_num, skip_existing=True):
    sub_id = f"sub-{sub_num}"

    # Skip if all output maps already exist for this subject
    if skip_existing:
        all_done = all(
            (OUT_DIR / f"{sub_id}_searchlight_{a}_vs_{b}.nii.gz").exists()
            for a, b in PAIRS
        )
        if all_done:
            print(f"=== {sub_id}: all maps already exist, skipping")
            return

    print(f"\n=== {sub_id}: building betas ===")
    t0 = datetime.now()
    betas_img, labels, runs = build_subject_betas(sub_num)
    if betas_img is None:
        print(f"  no data"); return
    print(f"  betas shape: {betas_img.shape}  ({datetime.now() - t0})")

    mask_path = find_brain_mask(sub_id, 1)
    if mask_path is None:
        print(f"  no brain mask, skipping"); return
    mask_img = nib.load(str(mask_path))

    for cond_a, cond_b in PAIRS:
        out_path = OUT_DIR / f"{sub_id}_searchlight_{cond_a}_vs_{cond_b}.nii.gz"
        if skip_existing and out_path.exists():
            print(f"  {cond_a} vs {cond_b}: exists, skipping"); continue

        sel = np.isin(labels, [cond_a, cond_b])
        if sel.sum() < 4:
            print(f"  skip {cond_a} vs {cond_b}: too few blocks"); continue

        print(f"  searchlight: {cond_a} vs {cond_b} ({sel.sum()} blocks)...")
        ts = datetime.now()

        sub_data = betas_img.get_fdata()[..., sel]
        sub_img  = nib.Nifti1Image(sub_data, betas_img.affine, betas_img.header)
        y        = labels[sel]
        groups   = runs[sel]

        clf = make_pipeline(StandardScaler(), LinearSVC(C=1.0, max_iter=5000))
        sl  = SearchLight(
            mask_img  = mask_img,
            radius    = RADIUS_MM,
            estimator = clf,
            n_jobs    = N_JOBS,
            cv        = LeaveOneGroupOut(),
            verbose   = 0,
        )
        sl.fit(sub_img, y, groups=groups)

        acc_data = sl.scores_
        acc_img  = nib.Nifti1Image(acc_data, betas_img.affine, betas_img.header)
        nib.save(acc_img, out_path)
        print(f"    saved {out_path.name}  (max acc = {acc_data.max():.3f}, "
              f"time = {datetime.now() - ts})")

## 6. (Optional) Smoke test — run on one subject first

Strongly recommended before kicking off the full batch. Verifies:
- Files are found correctly
- Beta extraction produces reasonable values (max accuracy should be ~0.7–1.0 in the most informative spheres, ~0.5 elsewhere)
- Time per subject is what you expect

In [ ]:
run_searchlight_subject(102)

## 7. Run the full batch (all 59 subjects)

In [ ]:
t0 = datetime.now()
for sub_num in GROUP_1_SUBS + GROUP_2_SUBS:
    try:
        run_searchlight_subject(sub_num)
    except Exception as e:
        print(f"  ERROR sub-{sub_num}: {type(e).__name__}: {e}")
print(f"\n=== Full batch done in {datetime.now() - t0} ===")

## 8. Quick check: what got written?

In [ ]:
rows = []
for grp, subs in [("BDD", GROUP_1_SUBS), ("HC", GROUP_2_SUBS)]:
    for s in subs:
        sub_id = f"sub-{s}"
        row = {"sub": s, "group": grp}
        for a, b in PAIRS:
            row[f"{a}_vs_{b}"] = (OUT_DIR / f"{sub_id}_searchlight_{a}_vs_{b}.nii.gz").exists()
        rows.append(row)
summary = pd.DataFrame(rows)
print(f"Subjects with all maps complete:")
complete_cols = [c for c in summary.columns if c not in ("sub", "group")]
summary["complete"] = summary[complete_cols].all(axis=1)
print(f"  BDD: {summary[(summary.group == 'BDD') & summary.complete].shape[0]}/{len(GROUP_1_SUBS)}")
print(f"  HC:  {summary[(summary.group == 'HC')  & summary.complete].shape[0]}/{len(GROUP_2_SUBS)}")
incomplete = summary[~summary.complete]
if len(incomplete):
    print("\nIncomplete subjects:")
    print(incomplete.to_string(index=False))

## 9. Second-level group analysis (BDD vs HC)

Once all subjects have searchlight maps, do the between-groups t-test on the per-subject accuracy maps. This produces a z-statistic map showing where decoding accuracy differs between groups, voxelwise. We then threshold and cluster-correct.

Run separately for each pair (each `(cond_a, cond_b)` produces its own group-level z-map).

In [ ]:
from nilearn.glm.second_level import SecondLevelModel
from nilearn.glm import threshold_stats_img

def group_compare(cond_a, cond_b, threshold_p=0.001, height_control="fpr"):
    """BDD vs HC group t-test on subject-level searchlight accuracy maps."""
    maps, design = [], []
    for s in GROUP_1_SUBS + GROUP_2_SUBS:
        p = OUT_DIR / f"sub-{s}_searchlight_{cond_a}_vs_{cond_b}.nii.gz"
        if not p.exists():
            print(f"  missing: {p.name}"); continue
        maps.append(str(p))
        design.append({"BDD": int(s < 200), "HC": int(s >= 200)})

    design_df = pd.DataFrame(design)
    print(f"Group sizes: BDD={design_df.BDD.sum()}, HC={design_df.HC.sum()}")

    slm = SecondLevelModel().fit(maps, design_matrix=design_df)

    # BDD - HC contrast (positive = BDD decodes better)
    z_bdd_minus_hc = slm.compute_contrast([1, -1], output_type="z_score")
    # HC - BDD contrast (positive = HC decodes better, i.e. BDD impaired)
    z_hc_minus_bdd = slm.compute_contrast([-1, 1], output_type="z_score")

    out_bdd_minus_hc = OUT_DIR / f"group_BDDminusHC_{cond_a}_vs_{cond_b}.nii.gz"
    out_hc_minus_bdd = OUT_DIR / f"group_HCminusBDD_{cond_a}_vs_{cond_b}.nii.gz"
    nib.save(z_bdd_minus_hc, out_bdd_minus_hc)
    nib.save(z_hc_minus_bdd, out_hc_minus_bdd)
    print(f"Saved: {out_bdd_minus_hc.name}")
    print(f"Saved: {out_hc_minus_bdd.name}")

    # Voxelwise threshold + cluster info (uncorrected p<0.001 by default)
    thr_img, threshold = threshold_stats_img(
        z_hc_minus_bdd, alpha=threshold_p, height_control=height_control,
        cluster_threshold=10
    )
    print(f"\nHC > BDD at p<{threshold_p} ({height_control}): "
          f"z >= {threshold:.2f}, min cluster = 10 voxels")
    return slm, z_bdd_minus_hc, z_hc_minus_bdd

# Run for the hypothesis-relevant pair
slm, z_bdd_hc, z_hc_bdd = group_compare("lowSF-lowC", "highSF-highC")

In [ ]:
# View the HC > BDD map (positive = decoding worse in BDD = hypothesis-consistent)
from nilearn import plotting

plotting.plot_stat_map(
    z_hc_bdd, threshold=3.1,                # ~p<0.001 uncorrected
    title="HC > BDD: searchlight decoding lowSF-lowC vs highSF-highC",
    display_mode="ortho", cut_coords=(0, -70, 0),
    cmap="hot"
)

In [ ]:
# Run the other pairs too
for cond_a, cond_b in PAIRS[1:]:
    print(f"\n=== {cond_a} vs {cond_b} ===")
    group_compare(cond_a, cond_b)

## 10. Cluster-corrected thresholding

For publication-grade thresholding, use cluster-extent FWE correction. Nilearn's `threshold_stats_img` with `height_control='fdr'` does FDR correction; for cluster-FWE you typically need a permutation-based approach (e.g., `randomise` from FSL or `nilearn.glm.second_level.non_parametric_inference`).

In [ ]:
from nilearn.glm.second_level import non_parametric_inference

def permutation_test(cond_a, cond_b, n_perm=1000):
    """Cluster-FWE-corrected group comparison via sign-flipping permutation."""
    maps, design = [], []
    for s in GROUP_1_SUBS + GROUP_2_SUBS:
        p = OUT_DIR / f"sub-{s}_searchlight_{cond_a}_vs_{cond_b}.nii.gz"
        if not p.exists():
            continue
        maps.append(str(p))
        design.append({"BDD": int(s < 200), "HC": int(s >= 200)})
    design_df = pd.DataFrame(design)

    print(f"Running {n_perm} permutations on {len(maps)} subjects...")
    out = non_parametric_inference(
        maps,
        design_matrix         = design_df,
        second_level_contrast = [-1, 1],          # HC - BDD
        n_perm                = n_perm,
        threshold             = 0.001,             # voxel-level p
        n_jobs                = -1,
    )
    out_path = OUT_DIR / f"group_HCminusBDD_{cond_a}_vs_{cond_b}_clusterFWE.nii.gz"
    nib.save(out["logp_max_size"], out_path)
    print(f"Saved cluster-FWE-corrected -log10(p): {out_path.name}")
    return out

# Run for the hypothesis-relevant pair (slow: ~10-30 min)
# perm_out = permutation_test("lowSF-lowC", "highSF-highC", n_perm=1000)